# event_datasets.duckdb Exploration

This notebook helps you inspect and explore the DuckDB database at `./data/event_datasets.duckdb`.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

In [2]:
# Resolve the project root from the notebook location.
project_root = Path.cwd().resolve().parent
db_path = project_root / "data" / "event_datasets.duckdb"

print(f"Project root: {project_root}")
print(f"DB path: {db_path}")
print(f"Exists: {db_path.exists()}")

Project root: /Users/6161138/uv_projects/teamnl-events-dataset
DB path: /Users/6161138/uv_projects/teamnl-events-dataset/data/event_datasets.duckdb
Exists: True


In [3]:
con = duckdb.connect(str(db_path), read_only=True)
con.execute("SELECT current_database()").fetchone()

('event_datasets',)

## Discover Schema

In [4]:
# List tables and views in the database.
con.execute("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,event_datasets,main,kg_datasets,"[dataset_id, title, alt_title, publication_dat...","[VARCHAR, VARCHAR, VARCHAR, DATE, DATE, DATE, ...",False
1,event_datasets,main,kg_variables,"[variable_id, dataset_id, variable_name, label...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False


In [5]:
# For each table, inspect schema and row count.
tables_df = con.execute("SHOW ALL TABLES").df()
table_names = tables_df["name"].dropna().tolist() if "name" in tables_df.columns else []

summary = []
for table in table_names:
    count = con.execute(f"SELECT COUNT(*) FROM \"{table}\"").fetchone()[0]
    summary.append({"table": table, "rows": count})

pd.DataFrame(summary).sort_values("rows", ascending=False).reset_index(drop=True)

,table,rows
0,kg_variables,63136
1,kg_datasets,578


In [7]:
# Pick one table to preview. Change selected_table as needed.
selected_table = table_names[1] if table_names else None
selected_table

'kg_variables'

In [8]:
if selected_table:
    display(con.execute(f"DESCRIBE \"{selected_table}\"").df())
    display(con.execute(f"SELECT * FROM \"{selected_table}\" LIMIT 20").df())
else:
    print("No tables found.")

,column_name,column_type,null,key,default,extra
0,variable_id,VARCHAR,YES,None,None,None
1,dataset_id,VARCHAR,YES,None,None,None
2,variable_name,VARCHAR,YES,None,None,None
3,label_,VARCHAR,YES,None,None,None
4,vocab_label,VARCHAR,YES,None,None,None
5,description,VARCHAR,YES,None,None,None
6,data_type,VARCHAR,YES,None,None,None
7,definition,VARCHAR,YES,None,None,None
8,num_predicates,VARCHAR,YES,None,None,None
9,tijdstip_predicates,VARCHAR[],YES,None,None,None


,variable_id,dataset_id,variable_name,label_,vocab_label,description,data_type,definition,num_predicates,tijdstip_predicates,valid_from,is_tijdstip
0,https://kg.odissei.nl/.well-known/genid/8bc351...,https://doi.org/10.57934/0B01E410803A2D15,MNDBEDRAGTWWW,Fiscaal maandbedrag TW-uitkeringen,Fiscaal maandbedrag TW-uitkeringen,Deze variabele wordt afgeleid door:\n\n1) alle...,Integer,Fiscaal bedrag aan uitkeringen in het kader va...,15,[],2017-01-01,False
1,https://kg.odissei.nl/.well-known/genid/94abd0...,https://doi.org/10.57934/0B01E4108079E7CB,SVOVRGHelpDipNiemand,Wie hebben je geholpen om te proberen toch je ...,Wie hebben je geholpen om te proberen toch je ...,NaN,Integer,Heeft een specifiek persoon of instantie een l...,13,[],2016-09-05,False
2,https://kg.odissei.nl/.well-known/genid/8c00b3...,https://doi.org/10.57934/0B01E410802D9C48,SILENQHB040,Day of household interview,Day of household interview,NaN,Integer,Datum waarop een enquête is ingevuld of afgeno...,13,[https://portal.odissei.nl/schema/variableInfo...,1989-01-01,True
3,https://kg.odissei.nl/.well-known/genid/947c07...,https://doi.org/10.57934/0B01E4108079E7CB,SVOREGLWOONaGevOpl,Wel/niet leerwegondersteunendonderwijs op 1 ok...,Geeft aan of men een indicatie heeft voor leer...,Met T wordt het jaar bedoeld waarin de dataver...,Integer,Heeft een persoon een indicatie voor leerwegon...,14,[],2016-09-05,False
4,https://kg.odissei.nl/.well-known/genid/448c36...,https://doi.org/10.57934/0B01E4108079E7CB,SVOVRGDiagnoseVsvGeenDeze,Van welke aandoeningen of ziekten is een diagn...,Van welke aandoeningen of ziekten is een diagn...,NaN,Integer,Is voor een specifieke ziekte of aandoening ee...,13,[],2014-01-01,False
5,https://kg.odissei.nl/.well-known/genid/6df73a...,https://doi.org/10.57934/0B01E41080723B03,SrtUniv,Soort universitaire huidige/aankomende opleidi...,Soort universitaire huidige/aankomende opleidi...,Wat voor soort universitaire opleiding (is dit...,Integer,Onderwijssoort/sector/opleiding/studierichting...,15,[],1899-12-31,False
6,https://kg.odissei.nl/.well-known/genid/fa10fe...,https://doi.org/10.57934/0B01E4108079E7CB,SVOAFLHoelangGel,Globaal aantal dagen geleden aanvang huidige baan,Globaal aantal dagen geleden aanvang huidige baan,NaN,Integer,Duidt aan hoe lang geleden een baan is aangeva...,13,[],2016-09-05,False
7,https://kg.odissei.nl/.well-known/genid/291f20...,https://doi.org/10.57934/0B01E41080723B03,DiagnoseEen,Voor één of meer aandoeningen of ziekten een d...,Voor één of meer aandoeningen of ziekten een d...,NaN,Integer,Diagnose gesteld door een arts of specialist v...,13,[],2021-05-01,False
8,https://kg.odissei.nl/.well-known/genid/1d87ff...,https://doi.org/10.57934/0B01E410805D0D51,RRINS,Recordidentificatienummer,Recordidentificatienummer,Willekeurig nummer voor intern gebruik bij Cen...,String,Een record identificerend nummer,14,[],2006-01-01,False
9,https://kg.odissei.nl/.well-known/genid/5b8b24...,https://doi.org/10.57934/0B01E410802154D6,RINOBJECTNUMMER,Adres-id van een persoon,Rinobjectnummer,Het nummer vormt in combinatie met 'Indicator ...,String,Dit nummer identificeert een geadresseerde ver...,15,[],1995-01-01,False


## Example Analysis Queries

In [9]:
# Top 20 tables by row count (computed using metadata).
pd.DataFrame(summary).sort_values("rows", ascending=False).head(20)

,table,rows
1,kg_variables,63136
0,kg_datasets,578


In [10]:
# Close the connection when done.
con.close()